In [0]:
if spark.catalog.tableExists("datamodeling.bronze.bronze_table"):
    last_load_date = spark.sql("SELECT max(last_updated) FROM datamodeling.bronze.bronze_table").collect()[0][0]
else:
    last_load_date = '1000-01-01'

last_load_date

In [0]:
spark.sql(f"""SELECT * FROM datamodeling.source.source_data
WHERE last_updated > '{last_load_date}'""").createOrReplaceTempView("bronze_source")

In [0]:
%sql
SELECT * FROM bronze_source

In [0]:
%sql
CREATE TABLE IF NOT EXISTS datamodeling.bronze.bronze_table (
    order_id INT,
    order_date DATE,
    customer_id INT,
    customer_name STRING,
    customer_email STRING,
    product_id INT,
    product_name STRING,
    product_category STRING,
    quantity INT,
    unit_price DECIMAL(10,2),
    payment_type STRING,
    country STRING,
    last_updated DATE,
    ingestion_ts TIMESTAMP
)
USING DELTA;

In [0]:
%sql
INSERT INTO datamodeling.bronze.bronze_table
SELECT
    order_id,
    order_date,
    customer_id,
    customer_name,
    customer_email,
    product_id,
    product_name,
    product_category,
    quantity,
    unit_price,
    payment_type,
    country,
    last_updated,
    ingestion_ts
FROM bronze_source;

In [0]:
%sql
Select * from datamodeling.bronze.bronze_table